# Solutions — `useState` & state management

Only look here after you've actually tried the exercises in `usestate.ipynb`.

### LESSON 25 — Exercise

**Part 1.**

In [ ]:
let l25calls = 0;

function l25counter() {
  l25calls = l25calls + 1;
  return l25calls;
}

function l25broken() {
  let calls = 0;
  calls = calls + 1;
  return calls;
}

const l25a = [1, 2, 3, 4, 5].map(() => l25counter());
const l25b = [1, 2, 3, 4, 5].map(() => l25broken());

console.log("outside the function:", l25a);
console.log("inside the function: ", l25b);

// l25broken demonstrates React's FIRST reason: local variables don't persist between
// renders - every call starts from 0 again.
//
// The second reason - that changing a variable does not trigger a render - cannot be
// demonstrated in plain JavaScript at all, because there is nothing here that renders.
// Only React can notice that it should call the component again, and only the setter
// tells it to.

**Part 2 — the playground.** Both additions use only what LESSON 25 gave you.

```jsx
function Counter({ title }) {
  const [count, setCount] = useState(0);
  const [clicks, setClicks] = useState(0);

  function handleStateClick() {
    setCount(count + 1);
    setClicks(clicks + 1);
  }

  function handleReset() {
    setCount(0);
    setClicks(clicks + 1);
  }

  return (
    <section>
      <h2>{title}</h2>
      <p>state: <b className="state">{count}</b></p>
      <p>clicks: <b className="clicks">{clicks}</b></p>
      <button className="state-btn" onClick={handleStateClick}>+1 (state)</button>
      <button onClick={handleReset}>Reset</button>
    </section>
  );
}
```

**3. What copy B shows.** `0` and `0`, however much you clicked in A.

Two calls to the same component are two **instances**, and each one gets its own state. A's
`count` and B's `count` are different values that happen to be created by the same line of
code — exactly like two calls to any function having their own arguments.

**Common mistakes.**

- Calling `useState` inside the handler instead of at the top of the component. Hooks have
  rules about where they may be called, and topic 13 covers them properly — for now, keep
  `useState` at the top of the component body, never inside a function, condition or loop.
- Writing `setCount(count++)`. `count++` tries to change `count` itself, which is a `const`,
  and it evaluates to the *old* number besides. `setCount(count + 1)` is the whole job.
- Adding a third state variable for "total clicks" in the parent to keep the two copies in
  step. That is a real need with a real answer, and it is LESSON 29 — resist for now.
- Expecting `clicks` to update if you only call `setCount`. Two state variables are two
  independent things; each one needs its own setter call.

### LESSON 25 — Mini challenge

| | what it is | what it can do |
|---|---|---|
| `let count = 0` inside the component | a normal local variable, created fresh on every render | hold a value *during one render*, and nothing more |
| `count` from `useState(0)` | the current value of this instance's state, handed back by React | be read and rendered; it is read-only from your side |
| `setCount` | the set function React gave you | store the next value **and** schedule a render |

**1. The error, and what would still be wrong.** `count` is declared with `const`, so
assigning to it throws `TypeError: Assignment to constant variable.`

Making it a `let` would not help. The value would still be thrown away when React called the
component again, and nothing would have told React to call it — both of React's reasons,
untouched. The `const` is not the obstacle; it is a guard rail in front of the obstacle.

**2. `setCount(0)` on a counter already at `0`.** React compares the new value with the
current one using `Object.is`, finds them identical, and "will skip re-rendering the
component and its children". Nothing on screen changes — and nothing needed to.

This is worth knowing for a reason beyond efficiency: a setter call is a *request*, not a
guarantee of a visible update.

**3. Two copies, one component.** Each `<Counter />` is a separate **instance**, and state is
local to an instance — so the two counters hold two entirely separate values created by the
same line of code.

**4. Why the parent cannot reset the child.** State is *fully private to the component
declaring it*; a parent has no way to reach inside and set it. What the parent can do is what
LESSON 16 showed: the child can accept a **function as a prop** and call it, or the parent can
pass down a value the child uses.

Which of those is the right answer — and where the state should have lived in the first place
— is LESSON 29, *lifting state up*. The mechanism you already have is the callback prop; what
is missing is the decision about who owns the value.

### LESSON 26 — Exercise

In [ ]:
// Conceptual model — not React's actual implementation.
function l26applyQueue(start, queue) {
  let value = start;
  for (const update of queue) {
    value = typeof update === "function" ? update(value) : update;
  }
  return value;
}

// Part 1
const l26queues = [
  ["[5]", [5]],
  ["[5, c => c + 1]", [5, (c) => c + 1]],
  ["[c => c + 1, 5]", [(c) => c + 1, 5]],
  ["[c => c * 2, c => c + 3]", [(c) => c * 2, (c) => c + 3]],
];

for (const [label, queue] of l26queues) {
  console.log(label.padEnd(26), "->", l26applyQueue(0, queue));
}

// The third one is the interesting case: the updater runs first and is then thrown away,
// because a plain value REPLACES whatever the queue had reached. Order matters because the
// two kinds of entry do different things - one calculates from what is there, the other
// overwrites it.

**Part 2 — the two handlers, starting from 10.**

In [ ]:
const l26start = 10;

const l26queueA = [l26start + 5, l26start + 5];               // both computed from the SAME 10
const l26queueB = [(c) => c + 5, (c) => c + 5];               // each fed the previous result

console.log("handleA ->", l26applyQueue(l26start, l26queueA));
console.log("handleB ->", l26applyQueue(l26start, l26queueB));

// After the last line of EITHER handler, `count` is still 10. The updater form changes the
// result React arrives at; it does not change the value the running handler can see.

**Common mistakes.**

- Expecting `handleA` to give `20`. Both lines read the same fixed `count`, so both queue the
  value `15`, and the second simply repeats the first.
- Reading the difference as "updaters are more correct". They are more correct *when the next
  value depends on the previous one*. `setCount(0)` written as `setCount(() => 0)` is noise.
- Thinking the updater form lets you read the new value in the handler. It does not — nothing
  does, until the next render.

### LESSON 26 — Mini challenge

**Part 1 — what the playground shows.**

1. The counter goes to **1**, and the handler reports:

```text
handler start — count is 0
handler end   — count is still 0
```

Three calls, one step, and the value visible to the handler never moved.

2. **Both render lines come after `handler end`.** React processed nothing until the handler
   had finished — *"React waits until all code in the event handlers has run before
   processing your state updates."* One handler, one render, regardless of how many setters
   were in it.

3. With three updater functions the counter goes to **3**, because each one is handed the
   result of the one before.

4. Two lines per render is `<StrictMode>` rendering the component an extra time in
   development (LESSON 3). It affects none of the numbers: the counter values and the
   ordering are the same either way. Count pairs of lines, not lines.

**Part 2 — judgement.**

1. **Clear filters → plain value.** `setCount(0)` does not depend on what was there.
2. **Like counter +1 → updater.** The next value is calculated from the previous one, and two
   quick clicks must not collapse into one.
3. **Set to the length of a prop list → plain value.** The next value comes from the prop, not
   from the current state.
4. **Double click adding 2 via a +1 helper twice → updater.** This is the exact case the
   lesson is about: two calls in one handler, each needing to build on the last. With the
   plain form both would compute from the same fixed value and the count would rise by 1.

### LESSON 27 — Exercise

**Part 1.** Every function returns a new object and touches nothing.

In [ ]:
const l27settings = {
  city: "Milan",
  notifications: true,
  display: { fontSize: 14, compact: false },
};

function l27withCity(settings, city) {
  return { ...settings, city };
}

function l27toggleNotifications(settings) {
  return { ...settings, notifications: !settings.notifications };
}

function l27withFontSize(settings, size) {
  return {
    ...settings,
    display: { ...settings.display, fontSize: size },
  };
}

const l27a = l27withCity(l27settings, "Bologna");
const l27b = l27toggleNotifications(l27settings);
const l27c = l27withFontSize(l27settings, 18);

console.log("new object? city  ", !Object.is(l27settings, l27a), "| city:", l27a.city, "| notifications kept:", l27a.notifications);
console.log("new object? toggle", !Object.is(l27settings, l27b), "| notifications:", l27b.notifications, "| city kept:", l27b.city);
console.log("new object? font  ", !Object.is(l27settings, l27c), "| copy:", l27c.display.fontSize, "| original:", l27settings.display.fontSize);
console.log("display replaced? ", !Object.is(l27settings.display, l27c.display));
console.log("compact preserved?", l27c.display.compact === false);

// The last two lines are the ones that matter. `withFontSize` had to spread TWICE: once for
// the settings object and once for `display`. Spreading only the outer level would have
// copied the REFERENCE to `display`, and `l27c.display.fontSize = size` would have changed
// the original too.

A cheaper check than reading the code: a nested update is correct when the original's nested
value is unchanged **and** the branch you did not touch is still there.

**Part 2 — the broken handler.**

In [ ]:
// What the user sees: nothing. Most likely nothing at all, and that is the honest answer.
//
//   account.plan = "pro";   <- changes the contents; `account` is still the same object
//   setAccount(account);    <- React compares with Object.is, sees the identical object,
//                              and skips the re-render
//
// Worse, the data really did change. So if anything else re-renders this component later,
// "pro" appears then, and the bug looks random.
//
// Corrected:
//
//   function handleUpgrade() {
//     setAccount((a) => ({ ...a, plan: "pro" }));
//   }
//
// The updater form is right because the next value is calculated FROM the previous one -
// every other field of the account has to survive, and the updater is handed the value
// React will actually apply it to (LESSON 26). Note the parentheses around the object.

console.log("broken:    same object, no render");
console.log("corrected: new object, render");

**Common mistakes.**

- Spreading only the outer level for a nested change: `{ ...settings, display: settings.display }`
  copies nothing useful — it is the same `display`, and editing it edits the original.
- Writing `setAccount((a) => { ...a, plan: "pro" })` without the parentheses. That is a
  function body, not an object; it returns `undefined` and wipes the state.
- Calling the setter but forgetting the spread — `setAccount({ plan: "pro" })` replaces the
  whole object, which is exactly what "replace" means. Every other field goes.
- Deciding that all mutation is forbidden. Building a fresh object field by field before
  handing it to the setter is fine; the rule is about objects already in state.

### LESSON 27 — Mini challenge

**1. Not a bug.** `next` is a brand-new object that nothing else references yet — React's
docs call this a *local mutation* and say it is completely okay. The object that must not be
touched is `profile`, and it wasn't. This is identical in effect to
`setProfile({ ...profile, name: "Ada" })`; pick whichever reads better.

**2. Everything except `city` is gone.** State becomes `{ city: "Bologna" }` — no `name`, no
`role`. "Replace" is literal: the setter stores the value you give it, it does not merge.
The spread is not decoration, it is what carries the other fields across. Expect this to
surface as `undefined` somewhere in the JSX rather than as an error.

**3. Something else re-rendered the component.** The direct edit did change the data, so a
later render — another piece of state, a parent updating, anything — reads the mutated
object and shows the new value. It is worse than a change that never appears because it
works *sometimes*: the code looks right, the bug is not reproducible on demand, and the
click that caused it is not the click that revealed it.

**4. `b` is not copied, and that is fine.** The new outer object holds the same `b` as
before. Nothing edits `b`, so nothing can go wrong — and sharing it is the reason a shallow
copy is fast. Shallowness is only a problem when you need to change what is inside a nested
object and forget to spread that level too.

### LESSON 28 — Exercise

**Part 1 — the drill.** Four functions, four one-liners. If yours are longer, they are
probably doing work `map` and `filter` already do.

In [ ]:
const l28list = [
  { id: "a", text: "Draft the plan", done: false },
  { id: "b", text: "Book the room", done: true },
  { id: "c", text: "Send the invites", done: false },
];

function l28add(list, id, text) {
  return [...list, { id, text, done: false }];
}

function l28remove(list, id) {
  return list.filter((t) => t.id !== id);
}

function l28toggle(list, id) {
  return list.map((t) => (t.id === id ? { ...t, done: !t.done } : t));
}

function l28rename(list, id, text) {
  return list.map((t) => (t.id === id ? { ...t, text } : t));
}

const l28withNew = l28add(l28list, "d", "Print the badges");
const l28shorter = l28remove(l28list, "b");
const l28flipped = l28toggle(l28list, "a");
const l28renamed = l28rename(l28list, "c", "Send the invitations");

console.log("add    ->", l28withNew.length, "items | original still", l28list.length);
console.log("remove ->", l28shorter.map((t) => t.id).join(" "));
console.log("rename ->", l28renamed[2].text, "| original still", l28list[2].text);

// The four proofs that matter:
console.log("new array?        ", !Object.is(l28list, l28flipped));
console.log("item a is new?    ", !Object.is(l28list[0], l28flipped[0]), "| done:", l28flipped[0].done);
console.log("items b, c shared?", Object.is(l28list[1], l28flipped[1]) && Object.is(l28list[2], l28flipped[2]));
console.log("original a intact?", l28list[0].done === false);

// Read those four together: the array is new, exactly one item is new, everything else is
// the SAME object, and the original never moved. That is "copies from the point of change
// all the way to the top level" - and nothing beyond it.

**Part 2 — three handlers, three bugs.**

In [ ]:
// 1 - addTask: nothing happens on screen.
//
//   tasks.push(...)  changes the array in place, so `tasks` is still the same array.
//   setTasks(tasks)  hands React the identical array; Object.is matches and the render
//                    is skipped. The task IS in the data - it just never appears.
//
//   Fix:  setTasks([...tasks, { id: nextId(), text, done: false }]);
//
// 2 - sortByText: nothing happens on screen, and the state is now scrambled anyway.
//
//   `sort` sorts in place AND returns the same array, so this has both problems at once:
//   the existing state was reordered behind React's back, and the setter got a value that
//   Object.is says is unchanged.
//
//   Fix:  setTasks(tasks.toSorted((a, b) => a.text.localeCompare(b.text)));
//   (or   setTasks([...tasks].sort((a, b) => a.text.localeCompare(b.text)));  )
//
// 3 - markFirstDone: the screen updates correctly. That is the problem.
//
//   [...tasks] really is a new array, so React re-renders and the row ticks. But
//   next[0] and tasks[0] are the SAME object, so the edit also changed the value that was
//   already in state. Nothing failed loudly, and now any code that compares the previous
//   state with the next one - an undo, a "what changed" check, a memoised child - is
//   comparing two references to one object that has already been changed.
//
//   Fix:  setTasks(tasks.map((t, i) => (i === 0 ? { ...t, done: true } : t)));
//
// Why 3 is worse: 1 and 2 fail visibly and immediately, so you find them in seconds.
// 3 works. It is found later, somewhere else, by someone debugging a different feature.

console.log("1 push        -> same array, no render");
console.log("2 sort        -> same array, no render, and state reordered");
console.log("3 shared item -> renders correctly, old state quietly changed");

**Common mistakes.**

- `setTasks(tasks.filter(...))` is right; `setTasks(tasks.splice(...))` is not. `splice`
  changes the array and returns *the removed items*, so the state becomes the deleted rows.
- Using `map` for removal by returning `null` or `undefined` for the unwanted item. The array
  keeps its length and you get empty rows. `filter` is the tool for removing.
- Spreading the object but not the array: `tasks[i] = { ...tasks[i], done: true }` builds a
  fresh object and then writes it into the array in state. `map` is what avoids that.
- Forgetting `id` and matching by index instead. It works until something is removed or
  reordered — exactly the LESSON 20 problem, on the data side this time.

### LESSON 28 — Mini challenge

**1. The split.**

| safe for state | not safe |
|---|---|
| `filter` · `map` · `concat` · `with` · `toSorted` | `push` · `splice` · `sort` · `arr[i] = x` |

The separating question is one thing only: **does it return a new array, or change the one I
gave it?** Every entry on the right hands you back the same array — which is precisely what
`Object.is` cannot tell apart from no change at all.

**2. Bug 3 — the shared item.** The checkbox row updates because the array really was new,
and that is exactly why it is hard to catch: the visible half is correct. What was also done,
invisibly, is an edit to the object that the *previous* state still points at. An undo that
kept a reference to the old array restores a list whose item has already been changed.

**3. Why sharing untouched items is a feature.** Two good answers:

- It is cheap. A list of 500 rows with one toggle creates one new object, not 500.
- It is information. Because unchanged items keep their identity, React — and later
  `React.memo`, in topic 23 — can tell at a glance which rows actually changed. Copying
  everything would erase that and make every row look new.

**4. Inserting at position 2.**

```js
const next = [...tasks.slice(0, 2), newTask, ...tasks.slice(2)];
```

`slice` copies a section and leaves the original alone, so the two halves plus the new item
build a complete new array. The copying twin does the same job in one call:

```js
const next = tasks.toSpliced(2, 0, newTask);
```

The method deliberately not used is `splice`, which would insert into the array already in
state and hand back the removed items rather than the list.

### LESSON 29 — Exercise

**Part 1.** Every field is calculated. `l29summary` holds nothing and remembers nothing.

In [ ]:
const l29list = [
  { id: "a", text: "Draft the plan", done: false, hours: 3 },
  { id: "b", text: "Book the room", done: true, hours: 1 },
  { id: "c", text: "Send the invites", done: false, hours: 2 },
];

function l29summary(list) {
  const done = list.filter((t) => t.done).length;
  return {
    total: list.length,
    done,
    remaining: list.length - done,
    hoursLeft: list.filter((t) => !t.done).reduce((sum, t) => sum + t.hours, 0),
    allDone: list.length > 0 && done === list.length,
  };
}

const l29after = l29list.map((t) => (t.id === "a" ? { ...t, done: true } : t));

console.log("before:", JSON.stringify(l29summary(l29list)));
console.log("after: ", JSON.stringify(l29summary(l29after)));

// Nothing had to be kept in step. The toggle changed the list; the summary followed because
// it is recalculated, not stored. In a component this function would simply be called during
// render, and `after` would be the new state.

`allDone` guards against the empty list on purpose — `done === list.length` is `true` for
zero tasks, which is almost never what a "you're all done!" banner should say.

**Part 1.3 — state or derived.**

In [ ]:
// STATE     the task list                   nothing else can produce it
// derived   the number of tasks left        list.filter(t => !t.done).length
// STATE     the text in the "new task" box  it is what the user typed; only they know it
//                                           (the mechanism for it is topic 10)
// derived   whether the list is empty       list.length === 0
// STATE     which filter is selected        a choice, not a calculation
// derived   the visible tasks               the LESSON 21 view: list.filter(...)
//
// The test each time: can I work this out from something I already hold? Two of these are
// genuine inputs - the data and the user's choices. Everything else falls out of them.

console.log("state: the list, the typed text, the selected filter");
console.log("derived: remaining, isEmpty, the visible tasks");

**Part 2 — the lift.** `Counter` no longer owns anything.

```jsx
function Counter({ title, count, onIncrement }) {
  return (
    <section>
      <h2>{title}</h2>
      <p>
        state: <b className="state">{count}</b>
      </p>
      <button className="state-btn" onClick={onIncrement}>
        +1 (state)
      </button>
    </section>
  );
}

export default function Experiment08() {
  const [count, setCount] = useState(0);

  function handleIncrement() {
    setCount((c) => c + 1);
  }

  return (
    <div>
      <h1>State</h1>
      <Counter title="A" count={count} onIncrement={handleIncrement} />
      <Counter title="B" count={count} onIncrement={handleIncrement} />
    </div>
  );
}
```

Click either button and both numbers move, because there is now only one number. The
`useState` import moves with the state — `Counter` no longer needs it.

`setCount((c) => c + 1)` rather than `setCount(count + 1)` because the next value is built
from the previous one (LESSON 26). One handler serves both copies; neither knows the other
exists.

**Own counts plus a total.** Two state variables in the parent, one per counter, and the
total derived:

```jsx
const [countA, setCountA] = useState(0);
const [countB, setCountB] = useState(0);
const total = countA + countB;          // derived - never a third useState
```

A third `useState` for the total is the exact mistake the first half of this lesson is about:
it would have to be updated in both handlers, and the first one you forget is the day it
starts lying.

**Common mistakes.**

- Lifting the state but leaving `useState` in the child too. Now there are two counts, the
  prop is ignored, and the buttons appear to do nothing to the shared number.
- Passing `onIncrement={handleIncrement()}` instead of `onIncrement={handleIncrement}`. That
  calls it during render — LESSON 22, and it bites again the moment handlers travel as props.
- Storing a derived value "for speed". `list.filter(...).length` on a screen's worth of rows
  costs nothing measurable; a stale badge costs an afternoon.
- Lifting state that only one component uses. It still works, which is why it spreads — but
  every level in between now carries a prop it has no use for.

### LESSON 29 — Mini challenge

**1. `items` and `isEmpty` will disagree.** Every place that changes `items` must remember to
call `setIsEmpty` too, and one of them eventually will not — most likely the third one added
six months later. The replacement is one line, and it cannot go stale:

```js
const isEmpty = items.length === 0;
```

**2. The state goes in their closest common parent.** All three siblings receive the value as
a prop; the one that changes it also receives a handler to call. The two that only read it
need nothing else — and that asymmetry is the point, because it says in the code which
component is allowed to change what.

**3. The cost of lifting everything.** Every component between the owner and the component
that actually uses the value has to accept props it does not care about and pass them on. The
tree becomes hard to move code around in — a component can no longer be relocated without
rethreading its props — and a change to any one value now re-renders from the top rather than
from where it matters. Topic 18 comes back to this problem with a tool for it; the fix until
then is simply not to lift what does not need lifting.

**4. Yes, still true.** A parent still cannot reach in and change a child's state — what
changed is that the value is no longer the child's state: the parent **owns** it now, and the
child only receives it and asks for changes.